## CAER-S Dataset preprocessing

Iterate over every image in every sub-folder and get its path as well as the sub-folder name.

In [14]:
from pathlib import Path
import csv

# Root folder containing all subfolders
dataset_dir = Path("/home/simon/Documents/Zwischen Wörtern und Pixeln/CAER-S_Dataset/CAER-S/test") #do the same for train later

# Output CSV file
output_csv = "CAER-S_split1.csv"

# Image extensions to include
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp"}

rows = []

# Iterate through all files recursively
for image_path in dataset_dir.rglob("*"):
    if image_path.is_file() and image_path.suffix.lower() in image_extensions:
        
        # Get immediate parent folder name
        subfolder_name = image_path.parent.name
        
        rows.append([
            str(image_path),   # full image path
            subfolder_name     # subfolder name
        ])

# Write to CSV
with open(output_csv, 
          mode = "w", 
          newline = "", 
          encoding = "utf-8") as f:
    
    writer = csv.writer(f)
    
    # Header
    writer.writerow(["image_path", "subfolder"])
    
    # Data rows
    writer.writerows(rows)

print(f"Saved {len(rows)} entries to {output_csv}")

Saved 20992 entries to CAER-S_split1.csv


## Create dataframes

In [15]:
import pandas as pd

#read in both splits
annotations_caer_split1 = pd.read_csv("/home/simon/Documents/Zwischen Wörtern und Pixeln/CAER-S_split1.csv")
annotations_caer_split2 = pd.read_csv("/home/simon/Documents/Zwischen Wörtern und Pixeln/CAER-S_split2.csv")

#merge dataframes
annotations_caer = pd.concat([annotations_caer_split1, annotations_caer_split2])

#reset index
annotations_caer.reset_index()

#verify
annotations_caer.info()
annotations_caer.head()

<class 'pandas.DataFrame'>
Index: 69999 entries, 0 to 49006
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   image_path  69999 non-null  str  
 1   subfolder   69999 non-null  str  
dtypes: str(2)
memory usage: 1.6 MB


,image_path,subfolder
0,/home/simon/Documents/Zwischen Wörtern und Pix...,Happy
1,/home/simon/Documents/Zwischen Wörtern und Pix...,Happy
2,/home/simon/Documents/Zwischen Wörtern und Pix...,Happy
3,/home/simon/Documents/Zwischen Wörtern und Pix...,Happy
4,/home/simon/Documents/Zwischen Wörtern und Pix...,Happy


## Recode emotion to fit other dataset

In [16]:
#check how many emotions exist
annotations_caer["subfolder"].unique()

<StringArray>
['Happy', 'Sad', 'Fear', 'Surprise', 'Neutral', 'Anger', 'Disgust', 'Angry']
Length: 8, dtype: str

In [18]:
#recode into four clusters
annotations_caer["emotion_recode2"] = annotations_caer["subfolder"].replace(
    ["Happy",
    "Sad",
    "Fear",
    "Neutral",
    "Anger",
    "Disgust",
    "Angry"],
    
    [0, #optimistic
    1, #pessimistic
    1, #pessimistic
    3, #neutral
    2, #hostile
    2, #hostile
    2] #hostile
)

#drop surprise
annotations_caer = annotations_caer[
    annotations_caer["emotion_recode2"] != "Surprise"]

#force integer to be sure
annotations_caer["emotion_recode2"].astype(int)

0        0
1        0
2        0
3        0
4        0
        ..
49002    2
49003    2
49004    2
49005    2
49006    2
Name: emotion_recode2, Length: 60000, dtype: int64

In [19]:
#verify clusters
annotations_caer.info()
annotations_caer.head()
annotations_caer["emotion_recode2"].unique()

<class 'pandas.DataFrame'>
Index: 60000 entries, 0 to 49006
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   image_path       60000 non-null  str   
 1   subfolder        60000 non-null  str   
 2   emotion_recode2  60000 non-null  object
dtypes: object(1), str(2)
memory usage: 1.8+ MB


array([0, 1, 3, 2], dtype=object)

In [20]:
annotations_caer.to_csv("annotations-caer_cleaned_4cluster.csv", 
                          index = False)